---
**Script name:** 04_02_02_10_simulation_export_results.ipynb

**Purpose of script:** This script is to export the results from the simulations.

**Author:** Dr Katie Allan

**Date Created:** 2022-07-10

**Email:** katie.allan@sussex.ac.uk

**License:** 

---

**Notes:** Change 'path_to_folder' to appropriate folder paths. Change 'postgresql://user:@localhost/db_name' to PostgreSQL databse login credentials.

---

## Import libraries that are required

In [ ]:
import pickle
import pandas as pd
import sqlalchemy as psql

## Functions that are used throughout the script

In [ ]:
# Assuming summary_stats has columns: sample_size, lower_fence, upper_fence
def calculate_whiskers(group, column):
    sample_size = group.name

    # Filter all_results to include only the current group
    group_results = all_results[all_results['sample_size'] == sample_size]

    # Lower whisker: closest value above the lower_fence
    lower_whisker = group_results[group_results[column] >= group['lower_fence'].iloc[0]].nsmallest(1, column)

    # Upper whisker: closest value below the upper_fence
    upper_whisker = group_results[group_results[column] <= group['upper_fence'].iloc[0]].nlargest(1, column)

    return pd.Series({
        'lower_whisker': lower_whisker[column].values[0] if not lower_whisker.empty else None,
        'upper_whisker': upper_whisker[column].values[0] if not upper_whisker.empty else None
    })

## Define variables

In [ ]:
# Define file directory and list of files
dir_path = os.getcwd()
filelist = os.listdir(dir_path)

## Load Data

In [ ]:
with open(f'{dir_path}/bathouse_1_sim_results.pickle', 'rb') as file:
    bathouse_1_results = pickle.load(file)

with open(f'{dir_path}/huxbear_sim_results.pickle', 'rb') as file:
    huxbear_results = pickle.load(file)

with open(f'{dir_path}/olchard_A_sim_results.pickle', 'rb') as file:
    olchard_A_results = pickle.load(file)

with open(f'{dir_path}/olchard_C_sim_results.pickle', 'rb') as file:
    olchard_C_results = pickle.load(file)

## Process Data

In [ ]:
#combine results from all four node simulations into one dataframe
dataframes = [bathouse_1_results['all_results']['last_mode_results'].assign(rec_id = 'bathouse_1'), huxbear_results['all_results']['last_mode_results'].assign(rec_id = 'huxbear'), olchard_A_results['all_results']['last_mode_results'].assign(rec_id = 'olchard_A'), olchard_C_results['all_results']['last_mode_results'].assign(rec_id = 'olchard_C')]

all_results = pd.concat(dataframes)
all_results

,time_diff,kl_div,error_estimate,abs_error_estimate,sample_size,set_seed,rec_id
0,2446 days 01:44:50,NaN,-8 days +11:56:57,7 days 12:03:03,100,seed_06,bathouse_1
1,2453 days 13:53:40,NaN,0 days 00:05:47,0 days 00:05:47,200,seed_06,bathouse_1
2,2453 days 13:48:40,NaN,0 days 00:00:47,0 days 00:00:47,300,seed_06,bathouse_1
3,2453 days 13:50:30,NaN,0 days 00:02:37,0 days 00:02:37,400,seed_06,bathouse_1
4,2453 days 13:57:10,NaN,0 days 00:09:17,0 days 00:09:17,500,seed_06,bathouse_1
...,...,...,...,...,...,...,...
595,2499 days 17:19:50,NaN,-1 days +23:59:52,0 days 00:00:08,2600,seed_97,olchard_C
596,2499 days 17:19:50,NaN,-1 days +23:59:52,0 days 00:00:08,2700,seed_97,olchard_C
597,2499 days 17:19:50,NaN,-1 days +23:59:52,0 days 00:00:08,2800,seed_97,olchard_C
598,2499 days 17:19:50,NaN,-1 days +23:59:52,0 days 00:00:08,2900,seed_97,olchard_C


In [ ]:
all_results['in_sec'] = all_results['abs_error_estimate'] / pd.Timedelta(seconds=1)
all_results

,time_diff,kl_div,error_estimate,abs_error_estimate,sample_size,set_seed,rec_id,in_sec
0,2446 days 01:44:50,NaN,-8 days +11:56:57,7 days 12:03:03,100,seed_06,bathouse_1,648183.0
1,2453 days 13:53:40,NaN,0 days 00:05:47,0 days 00:05:47,200,seed_06,bathouse_1,347.0
2,2453 days 13:48:40,NaN,0 days 00:00:47,0 days 00:00:47,300,seed_06,bathouse_1,47.0
3,2453 days 13:50:30,NaN,0 days 00:02:37,0 days 00:02:37,400,seed_06,bathouse_1,157.0
4,2453 days 13:57:10,NaN,0 days 00:09:17,0 days 00:09:17,500,seed_06,bathouse_1,557.0
...,...,...,...,...,...,...,...,...
595,2499 days 17:19:50,NaN,-1 days +23:59:52,0 days 00:00:08,2600,seed_97,olchard_C,8.0
596,2499 days 17:19:50,NaN,-1 days +23:59:52,0 days 00:00:08,2700,seed_97,olchard_C,8.0
597,2499 days 17:19:50,NaN,-1 days +23:59:52,0 days 00:00:08,2800,seed_97,olchard_C,8.0
598,2499 days 17:19:50,NaN,-1 days +23:59:52,0 days 00:00:08,2900,seed_97,olchard_C,8.0


In [ ]:
temp = all_results.sort_values(["sample_size", "in_sec"])

temp["prop_used"] = (
    temp
    .groupby("sample_size")
    .cumcount() + 1
) / (
    temp
    .groupby("sample_size")["in_sec"].transform("size")
)


In [ ]:
summary_stats_signed = (
    all_results
    .groupby(["rec_id", "sample_size"])
    .agg(
        error_mean=('error_estimate', 'mean'),
        error_median=('error_estimate', 'median'),
        error_std=('error_estimate', 'std'),
        error_min=('error_estimate', 'min'),
        error_max=('error_estimate', 'max')
    )
    .reset_index()
)

summary_stats_signed = summary_stats_signed[summary_stats_signed['sample_size'] > 2000]
summary_stats_signed

,rec_id,sample_size,error_mean,error_median,error_std,error_min,error_max
20,bathouse_1,2100,-1 days +23:59:44,-1 days +23:57:57,0 days 00:03:23.834298028,-1 days +23:51:37,0 days 00:07:17
21,bathouse_1,2200,-1 days +23:59:45.500000,-1 days +23:59:57,0 days 00:02:41.710429960,-1 days +23:51:37,0 days 00:03:17
22,bathouse_1,2300,0 days 00:00:34.500000,0 days 00:01:07,0 days 00:02:17.146675228,-1 days +23:57:47,0 days 00:04:47
23,bathouse_1,2400,-1 days +23:59:37,-1 days +23:58:57,0 days 00:02:03.288280059,-1 days +23:57:47,0 days 00:03:17
24,bathouse_1,2500,-1 days +23:59:08.500000,-1 days +23:57:57,0 days 00:02:46.426054387,-1 days +23:51:37,0 days 00:03:17
25,bathouse_1,2600,-1 days +23:59:20,-1 days +23:58:57,0 days 00:02:45.945457114,-1 days +23:51:37,0 days 00:02:37
26,bathouse_1,2700,-1 days +23:59:56,-1 days +23:59:57,0 days 00:02:49.236676378,-1 days +23:51:37,0 days 00:04:47
27,bathouse_1,2800,0 days 00:00:20.500000,-1 days +23:59:57,0 days 00:02:12.834483474,-1 days +23:57:47,0 days 00:04:47
28,bathouse_1,2900,-1 days +23:59:05.500000,-1 days +23:57:57,0 days 00:01:50.561961670,-1 days +23:57:47,0 days 00:02:37
29,bathouse_1,3000,-1 days +23:59:44,-1 days +23:59:57,0 days 00:01:54.712455158,-1 days +23:57:47,0 days 00:02:37


In [ ]:
summary_stats_absolute = (
    all_results
    .groupby(["rec_id", "sample_size"])
    .agg(
        error_mean=('abs_error_estimate', 'mean'),
        error_median=('abs_error_estimate', 'median'),
        error_std=('abs_error_estimate', 'std'),
        error_min=('abs_error_estimate', 'min'),
        error_max=('abs_error_estimate', 'max')
    )
    .reset_index()
)

summary_stats_absolute = summary_stats_absolute[summary_stats_absolute['sample_size'] > 2000]
summary_stats_absolute

,rec_id,sample_size,error_mean,error_median,error_std,error_min,error_max
20,bathouse_1,2100,0 days 00:02:42.600000,0 days 00:02:13,0 days 00:01:58.269273493,0 days 00:00:03,0 days 00:08:23
21,bathouse_1,2200,0 days 00:01:52.400000,0 days 00:02:03,0 days 00:01:54.336714184,0 days 00:00:03,0 days 00:08:23
22,bathouse_1,2300,0 days 00:02:05.200000,0 days 00:02:13,0 days 00:00:59.683022358,0 days 00:00:03,0 days 00:04:47
23,bathouse_1,2400,0 days 00:01:48.200000,0 days 00:02:13,0 days 00:00:58.595491114,0 days 00:00:03,0 days 00:03:17
24,bathouse_1,2500,0 days 00:02:16.700000,0 days 00:02:13,0 days 00:01:44.014725678,0 days 00:00:03,0 days 00:08:23
25,bathouse_1,2600,0 days 00:02:14.600000,0 days 00:02:13,0 days 00:01:40.754208479,0 days 00:00:03,0 days 00:08:23
26,bathouse_1,2700,0 days 00:02:03.300000,0 days 00:02:03,0 days 00:01:52.493789302,0 days 00:00:03,0 days 00:08:23
27,bathouse_1,2800,0 days 00:01:51.800000,0 days 00:02:13,0 days 00:01:10.215907629,0 days 00:00:03,0 days 00:04:47
28,bathouse_1,2900,0 days 00:01:53,0 days 00:02:08,0 days 00:00:43.697524920,0 days 00:00:03,0 days 00:02:37
29,bathouse_1,3000,0 days 00:01:39.900000,0 days 00:02:03,0 days 00:00:54.066138834,0 days 00:00:03,0 days 00:02:37


In [ ]:
#calculate summary statistics of signed error
summary_stats_signed = all_results[["sample_size", "error_estimate"]].groupby("sample_size").describe()
summary_stats_signed

error_estimate                            \
                     count                      mean   
sample_size                                            
100                     80    2 days 09:29:29.250000   
200                     80    3 days 16:15:16.750000   
300                     80    2 days 18:40:47.250000   
400                     80    0 days 06:49:15.375000   
500                     80    1 days 16:44:47.750000   
600                     80    1 days 11:03:24.375000   
700                     80    2 days 17:33:56.125000   
800                     80    1 days 10:58:28.500000   
900                     80    1 days 12:12:01.500000   
1000                    80           2 days 19:26:41   
1100                    80    1 days 07:55:37.375000   
1200                    80    2 days 19:43:45.375000   
1300                    80    0 days 11:18:19.750000   
1400                    80    3 days 02:52:05.625000   
1500                    80    0 days 20:00:25.375000   
1600                    80    2 days 09:20:04.375000   
1700                    80    1 days 15:41:15.625000   
1800                    80    0 days 08:14:05.750000   
1900                    80           0 days 23:30:12   
2000                    80    0 days 08:14:11.750000   
2100                    80  -1 days +23:58:06.500000   
2200                    80  -1 days +23:58:31.125000   
2300                    80  -1 days +23:58:26.750000   
2400                    80  -1 days +23:58:01.125000   
2500                    80  -1 days +23:58:30.375000   
2600                    80  -1 days +23:58:41.500000   
2700                    80  -1 days +23:58:29.750000   
2800                    80  -1 days +23:58:18.250000   
2900                    80  -1 days +23:58:11.625000   
3000                    80  -1 days +23:58:29.375000   

                                                             \
                                    std                 min   
sample_size                                                   
100          11 days 08:38:48.170825853  -12 days +00:09:15   
200          12 days 01:17:44.635379431  -12 days +00:09:15   
300           9 days 06:33:07.556584765  -11 days +00:21:25   
400           3 days 06:24:45.417919495  -13 days +00:10:35   
500           8 days 12:50:58.687487696   -3 days +00:25:28   
600           6 days 14:07:02.203504831   -1 days +00:08:42   
700           9 days 03:50:44.054227067   -1 days +23:49:17   
800           6 days 20:55:20.018873491   -1 days +23:50:38   
900           7 days 07:05:26.591241457   -1 days +23:50:38   
1000         10 days 00:15:12.205867440   -2 days +23:54:58   
1100          6 days 19:17:41.128901439   -1 days +23:50:38   
1200          9 days 18:17:24.583278073   -1 days +23:50:38   
1300          4 days 05:16:01.510194964   -1 days +23:50:38   
1400         10 days 06:07:55.403772153   -1 days +23:50:38   
1500          5 days 07:47:00.897153139   -1 days +23:50:38   
1600          9 days 07:36:29.698033685   -1 days +23:50:38   
1700          7 days 07:59:40.243601100   -1 days +23:50:35   
1800          3 days 01:54:42.124463815   -1 days +23:50:38   
1900          6 days 03:48:06.467691345   -1 days +23:50:38   
2000          3 days 01:54:41.430730676   -1 days +23:50:38   
2100          0 days 00:03:16.855019580   -1 days +23:50:38   
2200          0 days 00:03:08.039140485   -1 days +23:50:38   
2300          0 days 00:02:59.355209400   -1 days +23:50:35   
2400          0 days 00:03:23.841547733   -1 days +23:50:38   
2500          0 days 00:02:44.853166350   -1 days +23:50:35   
2600          0 days 00:02:52.167860391   -1 days +23:50:38   
2700          0 days 00:02:39.759114555   -1 days +23:51:05   
2800          0 days 00:02:43.706088904   -1 days +23:51:05   
2900          0 days 00:02:30.223830889   -1 days +23:51:05   
3000          0 days 00:02:22.871490011   -1 days +23:50:35   

                                                                 \
                

In [ ]:
#calculate summary statistics of absolute error
summary_stats_absolute = all_results[["sample_size", "abs_error_estimate"]].groupby("sample_size").describe()
summary_stats_absolute

abs_error_estimate                          \
                         count                    mean   
sample_size                                              
100                         80  3 days 21:56:24.425000   
200                         80  4 days 15:43:57.975000   
300                         80  3 days 11:13:48.425000   
400                         80  0 days 15:53:27.750000   
500                         80  1 days 19:11:17.650000   
600                         80  1 days 11:41:20.475000   
700                         80  2 days 17:36:40.300000   
800                         80  1 days 11:00:41.825000   
900                         80  1 days 12:15:46.300000   
1000                        80  2 days 20:06:08.975000   
1100                        80  1 days 07:59:10.575000   
1200                        80  2 days 19:46:13.250000   
1300                        80  0 days 11:22:29.350000   
1400                        80  3 days 02:55:59.750000   
1500                        80  0 days 20:04:08.750000   
1600                        80  2 days 09:24:21.400000   
1700                        80  1 days 15:45:13.025000   
1800                        80  0 days 08:18:36.650000   
1900                        80  0 days 23:34:04.675000   
2000                        80  0 days 08:18:24.500000   
2100                        80  0 days 00:02:41.075000   
2200                        80  0 days 00:02:32.200000   
2300                        80  0 days 00:02:27.350000   
2400                        80  0 days 00:02:34.425000   
2500                        80  0 days 00:02:06.600000   
2600                        80  0 days 00:02:09.650000   
2700                        80  0 days 00:02:10.275000   
2800                        80  0 days 00:02:20.975000   
2900                        80  0 days 00:02:16.650000   
3000                        80  0 days 00:02:00.300000   

                                                          \
                                    std              min   
sample_size                                                
100          10 days 22:11:46.585101858  0 days 00:00:17   
200          11 days 16:57:27.902691176  0 days 00:00:08   
300           9 days 00:50:02.312875351  0 days 00:00:02   
400           3 days 05:04:15.406359588  0 days 00:00:03   
500           8 days 12:20:32.851225566  0 days 00:00:03   
600           6 days 13:58:26.331027409  0 days 00:00:03   
700           9 days 03:49:54.452957121  0 days 00:00:03   
800           6 days 20:54:51.371276372  0 days 00:00:08   
900           7 days 07:04:39.482632228  0 days 00:00:03   
1000         10 days 00:03:55.498325411  0 days 00:00:08   
1100          6 days 19:16:58.876165201  0 days 00:00:08   
1200          9 days 18:16:41.279795505  0 days 00:00:08   
1300          4 days 05:15:33.204427879  0 days 00:00:03   
1400         10 days 06:06:43.252111034  0 days 00:00:03   
1500          5 days 07:46:25.424383483  0 days 00:00:03   
1600          9 days 07:35:22.916686988  0 days 00:00:08   
1700          7 days 07:58:45.983651914  0 days 00:00:03   
1800          3 days 01:54:11.418449069  0 days 00:00:03   
1900          6 days 03:47:28.946691551  0 days 00:00:03   
2000          3 days 01:54:12.784979202  0 days 00:00:03   
2100          0 days 00:02:39.761196394  0 days 00:00:03   
2200          0 days 00:02:21.065242110  0 days 00:00:03   
2300          0 days 00:02:17.793562341  0 days 00:00:03   
2400          0 days 00:02:58.080253207  0 days 00:00:03   
2500          0 days 00:02:18.131986048  0 days 00:00:03   
2600          0 days 00:02:17.332185280  0 days 00:00:03   
2700          0 days 00:02:08.781728301  0 days 00:00:03   
2800          0 days 00:02:10.989609695  0 days 00:00:03   
2900          0 days 00:02:04.705333699  0 days 00:00:03   
3000          0 days 00:01:58.633186383  0 days 00:00:03   

                                                             \
                                25%                  

In [ ]:
#flatten the dataframe to remove multiindexxing
summary_stats_signed.columns = summary_stats_signed.columns.get_level_values(1)
summary_stats_signed = summary_stats_signed.reset_index()
summary_stats_signed

,sample_size,count,mean,std,min,25%,50%,75%,max
0,100,80,2 days 09:29:29.250000,11 days 08:38:48.170825853,-12 days +00:09:15,0 days 00:00:37.750000,0 days 00:13:35,0 days 17:15:35.750000,61 days 20:08:27
1,200,80,3 days 16:15:16.750000,12 days 01:17:44.635379431,-12 days +00:09:15,-1 days +23:56:05.250000,0 days 00:03:35,1 days 00:19:17.500000,61 days 20:25:07
2,300,80,2 days 18:40:47.250000,9 days 06:33:07.556584765,-11 days +00:21:25,-1 days +23:57:16.750000,0 days 00:02:32,0 days 00:36:12,34 days 07:35:47
3,400,80,0 days 06:49:15.375000,3 days 06:24:45.417919495,-13 days +00:10:35,-1 days +23:57:27.750000,0 days 00:01:22,0 days 00:06:00.500000,25 days 16:05:07
4,500,80,1 days 16:44:47.750000,8 days 12:50:58.687487696,-3 days +00:25:28,-1 days +23:57:05,0 days 00:01:22,0 days 00:13:19,60 days 23:46:47
5,600,80,1 days 11:03:24.375000,6 days 14:07:02.203504831,-1 days +00:08:42,-1 days +23:57:56.500000,0 days 00:02:11,0 days 00:08:47,37 days 16:25:07
6,700,80,2 days 17:33:56.125000,9 days 03:50:44.054227067,-1 days +23:49:17,-1 days +23:57:56.500000,0 days 00:01:32,0 days 00:10:17,37 days 16:25:07
7,800,80,1 days 10:58:28.500000,6 days 20:55:20.018873491,-1 days +23:50:38,-1 days +23:57:57,0 days 00:01:32,0 days 00:03:22,38 days 20:14:17
8,900,80,1 days 12:12:01.500000,7 days 07:05:26.591241457,-1 days +23:50:38,-1 days +23:56:35,0 days 00:00:19.500000,0 days 00:02:34.500000,39 days 04:44:17
9,1000,80,2 days 19:26:41,10 days 00:15:12.205867440,-2 days +23:54:58,-1 days +23:56:28,0 days 00:00:15,0 days 00:02:37,39 days 04:44:17


In [ ]:
#flatten the dataframe to remove multiindexxing
summary_stats_absolute.columns = summary_stats_absolute.columns.get_level_values(1)
summary_stats_absolute = summary_stats_absolute.reset_index()
summary_stats_absolute

,sample_size,count,mean,std,min,25%,50%,75%,max
0,100,80,3 days 21:56:24.425000,10 days 22:11:46.585101858,0 days 00:00:17,0 days 00:05:22.750000,0 days 00:43:22.500000,3 days 23:25:09.750000,61 days 20:08:27
1,200,80,4 days 15:43:57.975000,11 days 16:57:27.902691176,0 days 00:00:08,0 days 00:03:51,0 days 00:12:52.500000,2 days 00:45:43,61 days 20:25:07
2,300,80,3 days 11:13:48.425000,9 days 00:50:02.312875351,0 days 00:00:02,0 days 00:02:34.250000,0 days 00:04:46,0 days 08:14:52,34 days 07:35:47
3,400,80,0 days 15:53:27.750000,3 days 05:04:15.406359588,0 days 00:00:03,0 days 00:01:37.250000,0 days 00:03:32,0 days 00:11:52.750000,25 days 16:05:07
4,500,80,1 days 19:11:17.650000,8 days 12:20:32.851225566,0 days 00:00:03,0 days 00:02:03,0 days 00:05:15,0 days 00:13:22,60 days 23:46:47
5,600,80,1 days 11:41:20.475000,6 days 13:58:26.331027409,0 days 00:00:03,0 days 00:02:04.500000,0 days 00:03:32,0 days 00:10:22,37 days 16:25:07
6,700,80,2 days 17:36:40.300000,9 days 03:49:54.452957121,0 days 00:00:03,0 days 00:01:34.250000,0 days 00:03:32,0 days 00:10:47.750000,37 days 16:25:07
7,800,80,1 days 11:00:41.825000,6 days 20:54:51.371276372,0 days 00:00:08,0 days 00:01:34.250000,0 days 00:02:42,0 days 00:05:26.750000,38 days 20:14:17
8,900,80,1 days 12:15:46.300000,7 days 07:04:39.482632228,0 days 00:00:03,0 days 00:01:51,0 days 00:03:06.500000,0 days 00:08:55,39 days 04:44:17
9,1000,80,2 days 20:06:08.975000,10 days 00:03:55.498325411,0 days 00:00:08,0 days 00:01:45,0 days 00:03:22,0 days 00:05:25.750000,39 days 04:44:17


In [ ]:
#Add interquartile range, min and max column
summary_stats_signed["IQR"] = summary_stats_signed["75%"] - summary_stats_signed["25%"]
summary_stats_signed["lower_fence"] = summary_stats_signed["25%"] - (1.5*summary_stats_signed["IQR"])
summary_stats_signed["upper_fence"] = summary_stats_signed["75%"] + (1.5*summary_stats_signed["IQR"])
summary_stats_signed

,sample_size,count,mean,std,min,25%,50%,75%,max,IQR,lower_fence,upper_fence
0,100,80,2 days 09:29:29.250000,11 days 08:38:48.170825853,-12 days +00:09:15,0 days 00:00:37.750000,0 days 00:13:35,0 days 17:15:35.750000,61 days 20:08:27,0 days 17:14:58,-2 days +22:08:10.750000,1 days 19:08:02.750000
1,200,80,3 days 16:15:16.750000,12 days 01:17:44.635379431,-12 days +00:09:15,-1 days +23:56:05.250000,0 days 00:03:35,1 days 00:19:17.500000,61 days 20:25:07,1 days 00:23:12.250000,-2 days +11:21:16.875000,2 days 12:54:05.875000
2,300,80,2 days 18:40:47.250000,9 days 06:33:07.556584765,-11 days +00:21:25,-1 days +23:57:16.750000,0 days 00:02:32,0 days 00:36:12,34 days 07:35:47,0 days 00:38:55.250000,-1 days +22:58:53.875000,0 days 01:34:34.875000
3,400,80,0 days 06:49:15.375000,3 days 06:24:45.417919495,-13 days +00:10:35,-1 days +23:57:27.750000,0 days 00:01:22,0 days 00:06:00.500000,25 days 16:05:07,0 days 00:08:32.750000,-1 days +23:44:38.625000,0 days 00:18:49.625000
4,500,80,1 days 16:44:47.750000,8 days 12:50:58.687487696,-3 days +00:25:28,-1 days +23:57:05,0 days 00:01:22,0 days 00:13:19,60 days 23:46:47,0 days 00:16:14,-1 days +23:32:44,0 days 00:37:40
5,600,80,1 days 11:03:24.375000,6 days 14:07:02.203504831,-1 days +00:08:42,-1 days +23:57:56.500000,0 days 00:02:11,0 days 00:08:47,37 days 16:25:07,0 days 00:10:50.500000,-1 days +23:41:40.750000,0 days 00:25:02.750000
6,700,80,2 days 17:33:56.125000,9 days 03:50:44.054227067,-1 days +23:49:17,-1 days +23:57:56.500000,0 days 00:01:32,0 days 00:10:17,37 days 16:25:07,0 days 00:12:20.500000,-1 days +23:39:25.750000,0 days 00:28:47.750000
7,800,80,1 days 10:58:28.500000,6 days 20:55:20.018873491,-1 days +23:50:38,-1 days +23:57:57,0 days 00:01:32,0 days 00:03:22,38 days 20:14:17,0 days 00:05:25,-1 days +23:49:49.500000,0 days 00:11:29.500000
8,900,80,1 days 12:12:01.500000,7 days 07:05:26.591241457,-1 days +23:50:38,-1 days +23:56:35,0 days 00:00:19.500000,0 days 00:02:34.500000,39 days 04:44:17,0 days 00:05:59.500000,-1 days +23:47:35.750000,0 days 00:11:33.750000
9,1000,80,2 days 19:26:41,10 days 00:15:12.205867440,-2 days +23:54:58,-1 days +23:56:28,0 days 00:00:15,0 days 00:02:37,39 days 04:44:17,0 days 00:06:09,-1 days +23:47:14.500000,0 days 00:11:50.500000


In [ ]:
#Add interquartile range, min and max column
summary_stats_absolute["IQR"] = summary_stats_absolute["75%"] - summary_stats_absolute["25%"]
summary_stats_absolute["lower_fence"] = summary_stats_absolute["25%"] - (1.5*summary_stats_absolute["IQR"])
summary_stats_absolute["upper_fence"] = summary_stats_absolute["75%"] + (1.5*summary_stats_absolute["IQR"])
summary_stats_absolute

,sample_size,count,mean,std,min,25%,50%,75%,max,IQR,lower_fence,upper_fence
0,100,80,3 days 21:56:24.425000,10 days 22:11:46.585101858,0 days 00:00:17,0 days 00:05:22.750000,0 days 00:43:22.500000,3 days 23:25:09.750000,61 days 20:08:27,3 days 23:19:47,-6 days +01:05:42.250000,9 days 22:24:50.250000
1,200,80,4 days 15:43:57.975000,11 days 16:57:27.902691176,0 days 00:00:08,0 days 00:03:51,0 days 00:12:52.500000,2 days 00:45:43,61 days 20:25:07,2 days 00:41:52,-4 days +23:01:03,5 days 01:48:31
2,300,80,3 days 11:13:48.425000,9 days 00:50:02.312875351,0 days 00:00:02,0 days 00:02:34.250000,0 days 00:04:46,0 days 08:14:52,34 days 07:35:47,0 days 08:12:17.750000,-1 days +11:44:07.625000,0 days 20:33:18.625000
3,400,80,0 days 15:53:27.750000,3 days 05:04:15.406359588,0 days 00:00:03,0 days 00:01:37.250000,0 days 00:03:32,0 days 00:11:52.750000,25 days 16:05:07,0 days 00:10:15.500000,-1 days +23:46:14,0 days 00:27:16
4,500,80,1 days 19:11:17.650000,8 days 12:20:32.851225566,0 days 00:00:03,0 days 00:02:03,0 days 00:05:15,0 days 00:13:22,60 days 23:46:47,0 days 00:11:19,-1 days +23:45:04.500000,0 days 00:30:20.500000
5,600,80,1 days 11:41:20.475000,6 days 13:58:26.331027409,0 days 00:00:03,0 days 00:02:04.500000,0 days 00:03:32,0 days 00:10:22,37 days 16:25:07,0 days 00:08:17.500000,-1 days +23:49:38.250000,0 days 00:22:48.250000
6,700,80,2 days 17:36:40.300000,9 days 03:49:54.452957121,0 days 00:00:03,0 days 00:01:34.250000,0 days 00:03:32,0 days 00:10:47.750000,37 days 16:25:07,0 days 00:09:13.500000,-1 days +23:47:44,0 days 00:24:38
7,800,80,1 days 11:00:41.825000,6 days 20:54:51.371276372,0 days 00:00:08,0 days 00:01:34.250000,0 days 00:02:42,0 days 00:05:26.750000,38 days 20:14:17,0 days 00:03:52.500000,-1 days +23:55:45.500000,0 days 00:11:15.500000
8,900,80,1 days 12:15:46.300000,7 days 07:04:39.482632228,0 days 00:00:03,0 days 00:01:51,0 days 00:03:06.500000,0 days 00:08:55,39 days 04:44:17,0 days 00:07:04,-1 days +23:51:15,0 days 00:19:31
9,1000,80,2 days 20:06:08.975000,10 days 00:03:55.498325411,0 days 00:00:08,0 days 00:01:45,0 days 00:03:22,0 days 00:05:25.750000,39 days 04:44:17,0 days 00:03:40.750000,-1 days +23:56:13.875000,0 days 00:10:56.875000


In [ ]:
#Change to timedelta
summary_stats_signed[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'IQR', 'lower_fence', 'upper_fence']] = summary_stats_signed[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'IQR', 'lower_fence', 'upper_fence']].apply(pd.to_timedelta)

In [ ]:
#Change to timedelta
summary_stats_absolute[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'IQR', 'lower_fence', 'upper_fence']] = summary_stats_absolute[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'IQR', 'lower_fence', 'upper_fence']].apply(pd.to_timedelta)

In [ ]:
# Apply the function to each group and merge the result into summary_stats
summary_stats_signed[['lower_whisker', 'upper_whisker']] = summary_stats_signed.groupby('sample_size', group_keys=False).apply(calculate_whiskers, column='error_estimate', include_groups=False).reset_index(drop=True)
summary_stats_signed

,sample_size,count,mean,std,min,25%,50%,75%,max,IQR,lower_fence,upper_fence,lower_whisker,upper_whisker
0,100,80,2 days 09:29:29.250000,11 days 08:38:48.170825853,-12 days +00:09:15,0 days 00:00:37.750000,0 days 00:13:35,0 days 17:15:35.750000,61 days 20:08:27,0 days 17:14:58,-2 days +22:08:10.750000,1 days 19:08:02.750000,-2 days +23:07:57,1 days 02:25:58
1,200,80,3 days 16:15:16.750000,12 days 01:17:44.635379431,-12 days +00:09:15,-1 days +23:56:05.250000,0 days 00:03:35,1 days 00:19:17.500000,61 days 20:25:07,1 days 00:23:12.250000,-2 days +11:21:16.875000,2 days 12:54:05.875000,-2 days +23:32:37,2 days 00:45:58
2,300,80,2 days 18:40:47.250000,9 days 06:33:07.556584765,-11 days +00:21:25,-1 days +23:57:16.750000,0 days 00:02:32,0 days 00:36:12,34 days 07:35:47,0 days 00:38:55.250000,-1 days +22:58:53.875000,0 days 01:34:34.875000,-1 days +23:06:57,0 days 01:06:47
3,400,80,0 days 06:49:15.375000,3 days 06:24:45.417919495,-13 days +00:10:35,-1 days +23:57:27.750000,0 days 00:01:22,0 days 00:06:00.500000,25 days 16:05:07,0 days 00:08:32.750000,-1 days +23:44:38.625000,0 days 00:18:49.625000,-1 days +23:48:12,0 days 00:15:07
4,500,80,1 days 16:44:47.750000,8 days 12:50:58.687487696,-3 days +00:25:28,-1 days +23:57:05,0 days 00:01:22,0 days 00:13:19,60 days 23:46:47,0 days 00:16:14,-1 days +23:32:44,0 days 00:37:40,-1 days +23:47:57,0 days 00:22:17
5,600,80,1 days 11:03:24.375000,6 days 14:07:02.203504831,-1 days +00:08:42,-1 days +23:57:56.500000,0 days 00:02:11,0 days 00:08:47,37 days 16:25:07,0 days 00:10:50.500000,-1 days +23:41:40.750000,0 days 00:25:02.750000,-1 days +23:50:38,0 days 00:17:07
6,700,80,2 days 17:33:56.125000,9 days 03:50:44.054227067,-1 days +23:49:17,-1 days +23:57:56.500000,0 days 00:01:32,0 days 00:10:17,37 days 16:25:07,0 days 00:12:20.500000,-1 days +23:39:25.750000,0 days 00:28:47.750000,-1 days +23:49:17,0 days 00:15:07
7,800,80,1 days 10:58:28.500000,6 days 20:55:20.018873491,-1 days +23:50:38,-1 days +23:57:57,0 days 00:01:32,0 days 00:03:22,38 days 20:14:17,0 days 00:05:25,-1 days +23:49:49.500000,0 days 00:11:29.500000,-1 days +23:50:38,0 days 00:08:47
8,900,80,1 days 12:12:01.500000,7 days 07:05:26.591241457,-1 days +23:50:38,-1 days +23:56:35,0 days 00:00:19.500000,0 days 00:02:34.500000,39 days 04:44:17,0 days 00:05:59.500000,-1 days +23:47:35.750000,0 days 00:11:33.750000,-1 days +23:50:38,0 days 00:08:47
9,1000,80,2 days 19:26:41,10 days 00:15:12.205867440,-2 days +23:54:58,-1 days +23:56:28,0 days 00:00:15,0 days 00:02:37,39 days 04:44:17,0 days 00:06:09,-1 days +23:47:14.500000,0 days 00:11:50.500000,-1 days +23:50:38,0 days 00:08:47


In [ ]:
# Apply the function to each group and merge the result into summary_stats
summary_stats_absolute[['lower_whisker', 'upper_whisker']] = summary_stats_absolute.groupby('sample_size', group_keys=False).apply(calculate_whiskers, column='abs_error_estimate', include_groups=False).reset_index(drop=True)
summary_stats_absolute

,sample_size,count,mean,std,min,25%,50%,75%,max,IQR,lower_fence,upper_fence,lower_whisker,upper_whisker
0,100,80,3 days 21:56:24.425000,10 days 22:11:46.585101858,0 days 00:00:17,0 days 00:05:22.750000,0 days 00:43:22.500000,3 days 23:25:09.750000,61 days 20:08:27,3 days 23:19:47,-6 days +01:05:42.250000,9 days 22:24:50.250000,0 days 00:00:17,7 days 12:03:03
1,200,80,4 days 15:43:57.975000,11 days 16:57:27.902691176,0 days 00:00:08,0 days 00:03:51,0 days 00:12:52.500000,2 days 00:45:43,61 days 20:25:07,2 days 00:41:52,-4 days +23:01:03,5 days 01:48:31,0 days 00:00:08,5 days 00:19:45
2,300,80,3 days 11:13:48.425000,9 days 00:50:02.312875351,0 days 00:00:02,0 days 00:02:34.250000,0 days 00:04:46,0 days 08:14:52,34 days 07:35:47,0 days 08:12:17.750000,-1 days +11:44:07.625000,0 days 20:33:18.625000,0 days 00:00:02,0 days 02:42:37
3,400,80,0 days 15:53:27.750000,3 days 05:04:15.406359588,0 days 00:00:03,0 days 00:01:37.250000,0 days 00:03:32,0 days 00:11:52.750000,25 days 16:05:07,0 days 00:10:15.500000,-1 days +23:46:14,0 days 00:27:16,0 days 00:00:03,0 days 00:22:22
4,500,80,1 days 19:11:17.650000,8 days 12:20:32.851225566,0 days 00:00:03,0 days 00:02:03,0 days 00:05:15,0 days 00:13:22,60 days 23:46:47,0 days 00:11:19,-1 days +23:45:04.500000,0 days 00:30:20.500000,0 days 00:00:03,0 days 00:22:17
5,600,80,1 days 11:41:20.475000,6 days 13:58:26.331027409,0 days 00:00:03,0 days 00:02:04.500000,0 days 00:03:32,0 days 00:10:22,37 days 16:25:07,0 days 00:08:17.500000,-1 days +23:49:38.250000,0 days 00:22:48.250000,0 days 00:00:03,0 days 00:17:07
6,700,80,2 days 17:36:40.300000,9 days 03:49:54.452957121,0 days 00:00:03,0 days 00:01:34.250000,0 days 00:03:32,0 days 00:10:47.750000,37 days 16:25:07,0 days 00:09:13.500000,-1 days +23:47:44,0 days 00:24:38,0 days 00:00:03,0 days 00:15:07
7,800,80,1 days 11:00:41.825000,6 days 20:54:51.371276372,0 days 00:00:08,0 days 00:01:34.250000,0 days 00:02:42,0 days 00:05:26.750000,38 days 20:14:17,0 days 00:03:52.500000,-1 days +23:55:45.500000,0 days 00:11:15.500000,0 days 00:00:08,0 days 00:09:22
8,900,80,1 days 12:15:46.300000,7 days 07:04:39.482632228,0 days 00:00:03,0 days 00:01:51,0 days 00:03:06.500000,0 days 00:08:55,39 days 04:44:17,0 days 00:07:04,-1 days +23:51:15,0 days 00:19:31,0 days 00:00:03,0 days 00:13:22
9,1000,80,2 days 20:06:08.975000,10 days 00:03:55.498325411,0 days 00:00:08,0 days 00:01:45,0 days 00:03:22,0 days 00:05:25.750000,39 days 04:44:17,0 days 00:03:40.750000,-1 days +23:56:13.875000,0 days 00:10:56.875000,0 days 00:00:08,0 days 00:09:22


## Export Data

In [ ]:
#send data to sql table
engine = psql.create_engine('postgresql://user:@localhost/db_name')
temp.to_sql('node_problem_error',
            con=engine,
            if_exists='replace',
            index=False)

C:\Users\allan\AppData\Local\Temp\ipykernel_16136\162683426.py:3: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.


400

In [ ]:
with open(f'{dir_path}simulation_error_estimate_results_summary_stats_signed.pickle', 'wb') as handle:
        pickle.dump(summary_stats_signed, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open(f'{dir_path}simulation_error_estimate_results_summary_stats_absolute.pickle', 'wb') as handle:
        pickle.dump(summary_stats_absolute, handle, protocol=pickle.HIGHEST_PROTOCOL)